# 07 — Auto-Retraining Loop with Multi-Signal Rewards (single H100)

This is the "set it and forget it" notebook. Hit **Run All** and a single H100 will:

1. Load the strongest open-source model that fits comfortably (default `Qwen3-14B`, auto-falls-back to `Qwen3-8B` on OOM).
2. Run a full SFT pass (Round 0) on the augmented BARC + RE-ARC + ARC pool.
3. Then loop:
   - Evaluate on a fixed held-out slice
   - Mine the tasks the model still gets wrong
   - DPO on `(chosen=truth, rejected=current model's wrong sample)` for those tasks
   - GRPO with a **multi-signal reward** (exact match + shape match + cell partial credit + format + length)
   - Re-evaluate; stop early if accuracy plateaus
4. Push the final adapter to HuggingFace Hub (optional) and emit a Kaggle-ready `submission.json`.

Every round writes a self-contained checkpoint to `runs/<RUN_ID>/r<N>/`, so you can resume from any point if something dies.

**Wall-clock estimate on H100 80GB:** ~12–24 h to convergence at the defaults below.


## What's different from notebook 05?

| | 05 (interp) | 07 (auto-loop) |
|---|---|---|
| Goal | *understand* the model with deep traces | *push the score* with automated improvement |
| Stages | base → SFT → DPO → GRPO (one shot) | SFT (round 0) + N×(DPO + GRPO) rounds |
| Hard-task mining | ❌ | ✅ each round |
| Reward signals | exact-match only | exact + shape + partial + format + length |
| Plateau early-stop | ❌ | ✅ |
| Per-round checkpoints | ❌ (one final adapter) | ✅ (one per round, resumable) |
| Model size | 8B | 14B (auto-falls back to 8B) |
| Forensic plotly viz | ✅ 11 views | minimal — just training curves |

Use **05** when you want to see what the model is doing inside.
Use **07** when you want to maximize the public-eval score.


In [ ]:
# 0.1 — Hardware check + helper
import torch, time, json, os, random, copy
from pathlib import Path
assert torch.cuda.is_available(), "Need a GPU. Switch to H100/A100 runtime."
GPU = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
IS_H100 = "H100" in GPU
print(f"GPU: {GPU} ({VRAM_GB:.0f} GB)")


In [ ]:
# 0.2 — Install (latest known-good for Qwen3 + GRPO)
!pip -q install -U \
    "transformers>=4.51,<4.55" "trl>=0.15,<0.20" "peft>=0.14" \
    "accelerate>=1.4" "bitsandbytes>=0.45" "datasets>=3.2" \
    "plotly>=5.24" sentencepiece huggingface_hub


In [ ]:
# 0.3 — Run config (the only knobs you should normally touch)
SEED = 0
random.seed(SEED); torch.manual_seed(SEED)

# Model: Qwen3-14B is the default. If we OOM at load time we fall back to 8B.
PRIMARY_MODEL  = "Qwen/Qwen3-14B"
FALLBACK_MODEL = "Qwen/Qwen3-8B"

# Loop config
N_ROUNDS         = 4         # round 0 = full SFT, then N_ROUNDS-1 DPO+GRPO rounds
HOLDOUT_N        = 80        # held-out eval tasks each round (subset of arc eval)
HARD_PER_ROUND   = 200       # cap on hard-task pairs per round
SFT_EPOCHS_R0    = 1
SFT_AUG_PER_TASK = 4         # multiplies pool size at SFT round 0
PLATEAU_DELTA    = 0.005     # stop if score gains < 0.5 pts for 2 rounds in a row
DPO_BETA         = 0.1
GRPO_BETA        = 0.04
GRPO_GROUPS      = 4

# Reward weights (composed into one scalar reward)
W_EXACT   = 1.0
W_SHAPE   = 0.2
W_PARTIAL = 0.5
W_FORMAT  = 0.1
W_LENGTH  = -0.05  # penalty if model rambles past expected output length

RUN_ID  = f"auto-{int(time.time())}"
ROOT    = Path("/content") if Path("/content").exists() else Path(".")
RUN_DIR = ROOT / "runs" / RUN_ID
DATA_DIR = ROOT / "arc_data"; DATA_DIR.mkdir(exist_ok=True)
for sub in ["logs", "eval"]: (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)
print("RUN_DIR:", RUN_DIR)


## Section 1 — Data: ARC + RE-ARC + BARC + augmentation


In [ ]:
# 1.1 — Download ARC-AGI-1
import urllib.request, zipfile, io
if not (DATA_DIR / "training").exists():
    with urllib.request.urlopen("https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip") as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(DATA_DIR)
    src = DATA_DIR / "ARC-AGI-master" / "data"
    for kind in ["training","evaluation"]:
        (DATA_DIR / kind).mkdir(exist_ok=True)
        for f in (src / kind).glob("*.json"):
            (DATA_DIR / kind / f.name).write_bytes(f.read_bytes())

arc_train = [json.loads(p.read_text()) for p in sorted((DATA_DIR/"training").glob("*.json"))]
arc_eval  = [(p.stem, json.loads(p.read_text())) for p in sorted((DATA_DIR/"evaluation").glob("*.json"))]
print(f"ARC: {len(arc_train)} train, {len(arc_eval)} eval")

# Held-out for eval each round (deterministic)
HOLDOUT = arc_eval[:HOLDOUT_N]


In [ ]:
# 1.2 — RE-ARC + BARC (best-effort, skip on failure)
re_arc_dir = DATA_DIR / "re_arc"
re_arc_tasks = []
if not re_arc_dir.exists():
    try:
        with urllib.request.urlopen("https://github.com/michaelhodel/re-arc/raw/main/re_arc.zip") as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(re_arc_dir)
    except Exception as e: print("RE-ARC skipped:", e)
if re_arc_dir.exists():
    for fp in sorted(re_arc_dir.rglob("*.json"))[:5000]:
        try: re_arc_tasks.append(json.loads(fp.read_text()))
        except: pass
print("RE-ARC:", len(re_arc_tasks))

from datasets import load_dataset, Dataset
try:
    barc = load_dataset(
        "barc0/200k_HEAVY_gpt4o-description-gpt4omini-code_generated_problems",
        split="train")
except Exception as e: barc = None; print("BARC skipped:", e)
print("BARC:", (len(barc) if barc else 0))


In [ ]:
# 1.3 — Augmentation + format helpers (lifted exactly from training notebooks)
def rot90(g):  return [list(r) for r in zip(*g[::-1])]
def fliplr(g): return [r[::-1] for r in g]
D4 = [lambda g: g, rot90,
      lambda g: rot90(rot90(g)),
      lambda g: rot90(rot90(rot90(g))),
      fliplr,
      lambda g: rot90(fliplr(g)),
      lambda g: rot90(rot90(fliplr(g))),
      lambda g: rot90(rot90(rot90(fliplr(g))))]

def apply_d4(task, k):
    f = D4[k]
    return {"train":[{"input":f(e["input"]),"output":f(e["output"])} for e in task["train"]],
            "test": [{"input":f(e["input"]),"output":f(e["output"])} for e in task["test"]]}

def apply_color_perm(task, perm):
    def remap(g): return [[perm[c] for c in row] for row in g]
    return {"train":[{"input":remap(e["input"]),"output":remap(e["output"])} for e in task["train"]],
            "test": [{"input":remap(e["input"]),"output":remap(e["output"])} for e in task["test"]]}

SYSTEM = ("You are an expert ARC puzzle solver. Output only the test output grid as digits "
          "separated by spaces, one row per line. No explanations.")

def grid_to_str(g): return "\n".join(" ".join(str(c) for c in row) for row in g)

def parse_grid(s):
    rows = []
    for line in s.strip().splitlines():
        line = line.strip()
        if not line: continue
        try: rows.append([int(c) for c in line.split()])
        except: return None
    if not rows or any(len(r) != len(rows[0]) for r in rows): return None
    return rows

def format_user(task):
    parts = ["You are solving an ARC puzzle. Each grid is a 2D array of digits 0-9 (colors).",
             "Find the transformation rule from the input/output examples, then apply it to the test input.", ""]
    for i, ex in enumerate(task["train"], 1):
        parts.append(f"Example {i} Input:");  parts.append(grid_to_str(ex["input"]))
        parts.append(f"Example {i} Output:"); parts.append(grid_to_str(ex["output"]))
        parts.append("")
    parts.append("Test Input:");  parts.append(grid_to_str(task["test"][0]["input"]))
    parts.append(""); parts.append("Test Output:")
    return "\n".join(parts)

def format_assistant(task):
    return grid_to_str(task["test"][0]["output"])

def _coerce_examples(exs):
    """Each ex must be {'input': 2D-list, 'output': 2D-list}.
    BARC variants encode pairs as dict, list[2], or tuple[2] — handle all."""
    if not isinstance(exs, list): return None
    out = []
    for e in exs:
        if isinstance(e, dict) and "input" in e and "output" in e:
            inp, outp = e["input"], e["output"]
        elif isinstance(e, (list, tuple)) and len(e) == 2:
            inp, outp = e[0], e[1]
        else:
            return None
        if not (isinstance(inp, list) and inp and isinstance(inp[0], list)): return None
        if not (isinstance(outp, list) and outp and isinstance(outp[0], list)): return None
        out.append({"input": inp, "output": outp})
    return out

def normalize_barc(row):
    train, test = None, None
    if "train" in row and "test" in row:
        train = _coerce_examples(row["train"]); test = _coerce_examples(row["test"])
    elif "examples" in row and "test" in row:
        train = _coerce_examples(row["examples"]); test = _coerce_examples(row["test"])
    elif "examples" in row:
        ex = _coerce_examples(row["examples"])
        if ex and len(ex) >= 2: train, test = ex[:-1], [ex[-1]]
    return {"train": train, "test": test} if (train and test) else None


In [ ]:
# 1.4 — Build augmented base pool
base_pool = list(arc_train) + list(re_arc_tasks)
if barc is not None:
    for r in barc.select(range(min(20000, len(barc)))):
        n = normalize_barc(r)
        if n and n["train"] and n["test"]: base_pool.append(n)
print(f"Base pool: {len(base_pool)}")

def gen_aug(tasks, n_per_task, rng):
    out = []
    for t in tasks:
        for _ in range(n_per_task):
            x = apply_d4(t, rng.randrange(8))
            perm = list(range(10)); rng.shuffle(perm)
            x = apply_color_perm(x, perm)
            out.append(x)
    return out

augmented = gen_aug(base_pool, n_per_task=SFT_AUG_PER_TASK, rng=random.Random(SEED))
print(f"Augmented samples: {len(augmented)}")


## Section 2 — The multi-signal reward function

Single function returns a scalar per completion. Composed of:

| Signal | Weight | Triggers when |
|---|---|---|
| **Exact match** | +1.0 | predicted grid equals truth |
| **Shape match** | +0.2 | HxW correct (regardless of cell values) |
| **Cell partial credit** | up to +0.5 | shape matches AND cells overlap |
| **Format reward** | +0.1 | output parses as a grid (rectangular, 0–9) |
| **Length penalty** | up to −0.05 | non-grid output > 50 tokens long (rambling) |

For a fully-correct prediction the total is ~1.8. For a totally wrong shape it's 0–0.1. GRPO normalizes this within each group so absolute scale doesn't matter.


In [ ]:
# 2.1 — Multi-signal reward
def multi_signal_reward(completions, prompts=None, ground_truth=None, **kw):
    truths = ground_truth if ground_truth is not None else [None] * len(completions)
    rewards = []
    for c, gt in zip(completions, truths):
        text = c if isinstance(c, str) else (c[0]["content"] if isinstance(c, list) else str(c))
        truth_grid = parse_grid(gt) if gt else None
        pred = parse_grid(text)
        r = 0.0
        if pred is not None:
            r += W_FORMAT
        if pred is None and len(text.split()) > 50:
            r += W_LENGTH
        if pred is not None and truth_grid is not None:
            same_shape = (len(pred) == len(truth_grid)
                          and len(pred[0]) == len(truth_grid[0]))
            if same_shape:
                r += W_SHAPE
                total = max(1, sum(len(row) for row in truth_grid))
                hits  = sum(1 for i in range(len(truth_grid))
                              for j in range(len(truth_grid[0]))
                              if pred[i][j] == truth_grid[i][j])
                r += W_PARTIAL * (hits / total)
                if pred == truth_grid:
                    r += W_EXACT
        rewards.append(r)
    return rewards


## Section 3 — Eval helper + hard-task miner

Each round we run the current model (greedy) on `HOLDOUT`, log the score, and collect every wrong task into a "hard tasks" pool that drives the next round of DPO + GRPO.


In [ ]:
# 3.1 — Eval + miner
def to_chat_prompt(tok, task, qwen3_kw):
    msgs = [{"role":"system","content":SYSTEM},
            {"role":"user","content":format_user(task)}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, **qwen3_kw)

@torch.inference_mode()
def predict_greedy(model, tok, task, max_new=2048, qwen3_kw=None):
    text = to_chat_prompt(tok, task, qwen3_kw or {})
    inp = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=max_new, do_sample=False,
                        pad_token_id=tok.eos_token_id)
    return parse_grid(tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)), text

def eval_holdout(model, tok, qwen3_kw, holdout=HOLDOUT):
    correct = 0; per_task = []
    for name, task in holdout:
        pred, _ = predict_greedy(model, tok, task, qwen3_kw=qwen3_kw)
        ok = (pred == task["test"][0]["output"])
        correct += int(ok)
        per_task.append({"task": name, "correct": ok, "pred": pred})
    acc = correct / max(1, len(holdout))
    return acc, per_task

def mine_hard_pairs(model, tok, qwen3_kw, holdout=HOLDOUT, cap=HARD_PER_ROUND):
    """For each task model gets wrong, sample once at temp=0.7 to get a 'rejected'
    completion, pair it against the ground-truth as 'chosen'."""
    pairs = []
    for name, task in holdout:
        if len(pairs) >= cap: break
        text = to_chat_prompt(tok, task, qwen3_kw)
        inp = tok(text, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            out = model.generate(**inp, max_new_tokens=512, do_sample=True,
                                 temperature=0.7, top_p=0.95,
                                 pad_token_id=tok.eos_token_id)
        sampled = tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()
        gt = format_assistant(task)
        sampled_grid = parse_grid(sampled)
        if sampled_grid != task["test"][0]["output"] and sampled.strip():
            pairs.append({"prompt": text, "chosen": gt, "rejected": sampled,
                          "ground_truth": gt, "task": name})
    return pairs


## Section 4 — Stage trainers

Three small functions: `do_sft`, `do_dpo`, `do_grpo`. Each takes the live model + a dataset, runs one mini-pass, saves the adapter for that round, returns the model.


In [ ]:
# 4.1 — SFT (used at round 0 only)
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig, GRPOTrainer, GRPOConfig
from transformers import TrainerCallback

class JsonlLogger(TrainerCallback):
    def __init__(self, path):
        self.path = path; self.f = open(path, "a")
    def on_log(self, args, state, control, logs=None, **kw):
        if logs is None: return
        rec = {"step": state.global_step, **{k: float(v) for k,v in logs.items()
                                              if isinstance(v,(int,float))}}
        self.f.write(json.dumps(rec) + "\n"); self.f.flush()

MAX_LEN = 3500

def do_sft(model, tok, qwen3_kw, sft_dataset, round_idx):
    out_dir = RUN_DIR / f"r{round_idx}" / "sft"
    out_dir.mkdir(parents=True, exist_ok=True)
    cfg = SFTConfig(
        output_dir=str(out_dir),
        num_train_epochs=SFT_EPOCHS_R0,
        per_device_train_batch_size=2 if VRAM_GB >= 70 else 1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        bf16=True,
        logging_steps=20,
        save_steps=10_000,
        save_total_limit=1,
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        max_seq_length=MAX_LEN,
        dataset_text_field="text",
        packing=False, report_to="none", seed=SEED,
    )
    trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=sft_dataset, args=cfg,
                         callbacks=[JsonlLogger(str(RUN_DIR / "logs" / f"r{round_idx}_sft.jsonl"))])
    trainer.train()
    trainer.model.save_pretrained(str(out_dir))
    return trainer.model

def do_dpo(model, tok, qwen3_kw, pairs, round_idx):
    if not pairs:
        print(f"[r{round_idx}] DPO: no pairs, skipping"); return model
    out_dir = RUN_DIR / f"r{round_idx}" / "dpo"
    out_dir.mkdir(parents=True, exist_ok=True)
    ds = Dataset.from_list(pairs)
    cfg = DPOConfig(
        output_dir=str(out_dir),
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=5e-5 / max(1, round_idx),  # decay LR over rounds
        lr_scheduler_type="cosine", warmup_ratio=0.03, bf16=True,
        logging_steps=10, save_steps=10_000, save_total_limit=1,
        optim="paged_adamw_8bit", gradient_checkpointing=True,
        beta=DPO_BETA,
        max_length=MAX_LEN, max_prompt_length=MAX_LEN-768,
        report_to="none", seed=SEED,
    )
    trainer = DPOTrainer(model=model, ref_model=None, args=cfg, train_dataset=ds, tokenizer=tok,
                         callbacks=[JsonlLogger(str(RUN_DIR / "logs" / f"r{round_idx}_dpo.jsonl"))])
    trainer.train()
    trainer.model.save_pretrained(str(out_dir))
    return trainer.model

def do_grpo(model, tok, qwen3_kw, pairs, round_idx):
    if not pairs:
        print(f"[r{round_idx}] GRPO: no prompts, skipping"); return model
    out_dir = RUN_DIR / f"r{round_idx}" / "grpo"
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = [{"prompt": p["prompt"], "ground_truth": p["ground_truth"]} for p in pairs]
    ds = Dataset.from_list(rows)
    cfg = GRPOConfig(
        output_dir=str(out_dir),
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=5e-6 / max(1, round_idx),
        lr_scheduler_type="cosine", warmup_ratio=0.03, bf16=True,
        logging_steps=5, save_steps=10_000, save_total_limit=1,
        optim="paged_adamw_8bit", gradient_checkpointing=True,
        num_generations=GRPO_GROUPS, max_prompt_length=MAX_LEN-512,
        max_completion_length=512, beta=GRPO_BETA,
        report_to="none", seed=SEED,
    )
    trainer = GRPOTrainer(model=model, reward_funcs=multi_signal_reward, args=cfg,
                          train_dataset=ds, processing_class=tok,
                          callbacks=[JsonlLogger(str(RUN_DIR / "logs" / f"r{round_idx}_grpo.jsonl"))])
    trainer.train()
    trainer.model.save_pretrained(str(out_dir))
    return trainer.model


## Section 5 — Round 0: load model + initial SFT

We try the bigger model first; if loading fails (rare: H100 should fit Qwen3-14B in 4-bit + LoRA), we fall back to 8B.


In [ ]:
# 5.1 — Load with auto-fallback
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

def load_with_lora(model_id):
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                            bnb_4bit_compute_dtype=torch.bfloat16,
                            bnb_4bit_use_double_quant=True)
    attn_impl = "flash_attention_2" if IS_H100 else "sdpa"
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    qwen3_kw = {"enable_thinking": False} if "Qwen3" in model_id else {}
    m = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb, device_map={"": 0},
        torch_dtype=torch.bfloat16, attn_implementation=attn_impl, trust_remote_code=True)
    m.config.use_cache = False
    m = prepare_model_for_kbit_training(m)
    lora = LoraConfig(r=64, lora_alpha=128, lora_dropout=0.05, bias="none",
                     task_type="CAUSAL_LM",
                     target_modules=["q_proj","k_proj","v_proj","o_proj",
                                     "gate_proj","up_proj","down_proj"])
    m = get_peft_model(m, lora)
    m.print_trainable_parameters()
    return m, tok, qwen3_kw

try:
    MODEL_ID = PRIMARY_MODEL
    model, tok, QWEN3_KW = load_with_lora(MODEL_ID)
except Exception as e:
    print(f"Falling back to {FALLBACK_MODEL} due to: {e}")
    torch.cuda.empty_cache()
    MODEL_ID = FALLBACK_MODEL
    model, tok, QWEN3_KW = load_with_lora(MODEL_ID)
print("ACTIVE MODEL:", MODEL_ID)


In [ ]:
# 5.2 — Tokenize SFT pool + run round-0 SFT
def to_chat_full(t):
    msgs = [{"role":"system","content":SYSTEM},
            {"role":"user","content":format_user(t)},
            {"role":"assistant","content":format_assistant(t)}]
    return tok.apply_chat_template(msgs, tokenize=False, **QWEN3_KW)

sft_texts = []
for t in augmented:
    s = to_chat_full(t)
    if len(tok(s, add_special_tokens=False).input_ids) <= MAX_LEN:
        sft_texts.append(s)
print(f"SFT samples after length filter: {len(sft_texts)}")
sft_ds = Dataset.from_dict({"text": sft_texts}).shuffle(seed=SEED)

model = do_sft(model, tok, QWEN3_KW, sft_ds, round_idx=0)


In [ ]:
# 5.3 — Round 0 eval
acc0, per_task0 = eval_holdout(model, tok, QWEN3_KW)
print(f"[r0] holdout acc: {acc0:.1%}")
(RUN_DIR / "eval" / "r0.json").write_text(json.dumps({"acc": acc0, "per_task": per_task0}, indent=2))
SCORE_HISTORY = [acc0]


## Section 6 — The auto-loop

For rounds 1..N-1, each iteration:

1. Mine hard tasks (held-out tasks the current model fails on, with one fresh sampled "rejected" completion per task).
2. DPO mini-pass on those `(chosen=truth, rejected=sample)` pairs.
3. GRPO mini-pass with the multi-signal reward on the same prompts.
4. Re-evaluate. If the last 2 rounds together gained < `PLATEAU_DELTA`, stop.

Each round's full state lives in `runs/<RUN_ID>/r<N>/{sft,dpo,grpo}/` so you can resume from any checkpoint.


In [ ]:
# 6.1 — The loop
import gc

def cleanup():
    gc.collect(); torch.cuda.empty_cache()

for rnd in range(1, N_ROUNDS):
    print(f"\n========== ROUND {rnd}/{N_ROUNDS-1} ==========")
    cleanup()

    # Mine hard pairs from current model on holdout
    pairs = mine_hard_pairs(model, tok, QWEN3_KW, holdout=HOLDOUT, cap=HARD_PER_ROUND)
    print(f"[r{rnd}] hard pairs mined: {len(pairs)}")
    cleanup()

    # DPO
    model = do_dpo(model, tok, QWEN3_KW, pairs, round_idx=rnd)
    cleanup()

    # GRPO with multi-signal reward
    model = do_grpo(model, tok, QWEN3_KW, pairs, round_idx=rnd)
    cleanup()

    # Eval
    acc, per_task = eval_holdout(model, tok, QWEN3_KW)
    SCORE_HISTORY.append(acc)
    print(f"[r{rnd}] holdout acc: {acc:.1%}  history: {[f'{s:.1%}' for s in SCORE_HISTORY]}")
    (RUN_DIR / "eval" / f"r{rnd}.json").write_text(json.dumps({
        "acc": acc, "per_task": per_task, "history": SCORE_HISTORY}, indent=2))

    # Plateau early-stop
    if rnd >= 2:
        gain_recent = SCORE_HISTORY[-1] - SCORE_HISTORY[-3]
        if gain_recent < 2 * PLATEAU_DELTA:
            print(f"[r{rnd}] plateau detected (gain over last 2 rounds = {gain_recent:.3%}), stopping early")
            break


## Section 7 — Final artifacts: training curves + push to Hub


In [ ]:
# 7.1 — Score history plot
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(SCORE_HISTORY))),
                         y=[s*100 for s in SCORE_HISTORY],
                         mode="lines+markers+text",
                         text=[f"{s:.1%}" for s in SCORE_HISTORY],
                         textposition="top center", line=dict(width=3)))
fig.update_layout(title=f"Auto-retrain score over rounds ({MODEL_ID})",
                  xaxis_title="round", yaxis_title="holdout accuracy (%)",
                  template="plotly_white", height=480)
fig.write_html(str(RUN_DIR / "score_history.html"))
fig.show()


In [ ]:
# 7.2 — Per-round per-phase loss curves
def load_jsonl(p):
    if not Path(p).exists(): return []
    return [json.loads(l) for l in Path(p).read_text().splitlines() if l.strip()]

phases = ["sft", "dpo", "grpo"]
fig = go.Figure()
for r in range(N_ROUNDS):
    for ph in phases:
        rows = load_jsonl(RUN_DIR / "logs" / f"r{r}_{ph}.jsonl")
        if not rows: continue
        x = [v["step"] for v in rows if "loss" in v]
        y = [v["loss"] for v in rows if "loss" in v]
        if y:
            fig.add_trace(go.Scatter(x=x, y=y, mode="lines",
                                     name=f"r{r}-{ph}"))
fig.update_layout(title="Loss curves per round x phase",
                  xaxis_title="step", yaxis_title="loss",
                  template="plotly_white", height=600)
fig.write_html(str(RUN_DIR / "loss_curves.html"))
fig.show()


In [ ]:
# 7.3 — Best-round adapter (highest holdout acc) -> 'final' alias
import shutil
best_round = max(range(len(SCORE_HISTORY)), key=lambda i: SCORE_HISTORY[i])
print(f"Best round: r{best_round} with acc={SCORE_HISTORY[best_round]:.1%}")

# Find the latest stage in that round (grpo > dpo > sft)
for stage in ["grpo", "dpo", "sft"]:
    src = RUN_DIR / f"r{best_round}" / stage
    if src.exists():
        dst = RUN_DIR / "final"
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"Best adapter: {src} -> {dst}")
        break


In [ ]:
# 7.4 — Push final adapter to HuggingFace Hub (optional)
PUSH_TO_HUB = False
HUB_REPO_ID = "your-handle/qwen3-arc-auto"

if PUSH_TO_HUB:
    from huggingface_hub import login, HfApi
    login()
    HfApi().create_repo(HUB_REPO_ID, repo_type="model", exist_ok=True, private=False)
    HfApi().upload_folder(folder_path=str(RUN_DIR / "final"),
                          repo_id=HUB_REPO_ID, repo_type="model")
    print(f"Pushed: https://huggingface.co/{HUB_REPO_ID}")
else:
    print("Skipped (set PUSH_TO_HUB=True + edit HUB_REPO_ID).")


## Section 8 — Final benchmark + submission.json

We score the **best round's adapter** on the full ARC eval (greedy + D4 vote) and emit a Kaggle-format submission.json.


In [ ]:
# 8.1 — D4-vote helper (same as notebook 06)
INV = [0, 3, 2, 1, 4, 5, 6, 7]
def predict_d4_vote(task):
    counts = {}
    for k in range(8):
        try:
            t_aug = apply_d4(task, k)
            pred, _ = predict_greedy(model, tok, t_aug, qwen3_kw=QWEN3_KW)
            if pred is None: continue
            inv = D4[INV[k]](pred)
            key = tuple(tuple(r) for r in inv)
            counts[key] = counts.get(key, 0) + 1
        except Exception: continue
    if not counts: return None
    return [list(r) for r in max(counts, key=counts.get)]


In [ ]:
# 8.2 — Full 400-task eval on best adapter
from tqdm.auto import tqdm
eval_full = arc_eval[:]
greedy_preds = {}; d4_preds = {}
g_correct = 0; d_correct = 0

for name, task in tqdm(eval_full, desc="final eval"):
    truth = task["test"][0]["output"]
    g, _ = predict_greedy(model, tok, task, qwen3_kw=QWEN3_KW)
    d = predict_d4_vote(task)
    greedy_preds[name] = g
    d4_preds[name] = d
    g_correct += int(g == truth)
    d_correct += int(d == truth)

n = len(eval_full)
print(f"\nGREEDY: {g_correct}/{n} = {g_correct/n:.1%}")
print(f"D4VOTE: {d_correct}/{n} = {d_correct/n:.1%}")
pass2 = sum(1 for name, t in eval_full
            if greedy_preds[name] == t["test"][0]["output"]
            or d4_preds[name]    == t["test"][0]["output"])
print(f"pass@2: {pass2}/{n} = {pass2/n:.1%}")

(RUN_DIR / "eval" / "final.json").write_text(json.dumps({
    "greedy": g_correct/n, "d4vote": d_correct/n, "pass2": pass2/n,
    "n": n, "best_round": best_round, "history": SCORE_HISTORY,
}, indent=2))


In [ ]:
# 8.3 — Submission.json (attempt_1 = D4 vote, attempt_2 = greedy)
submission = {}
for name, task in eval_full:
    a1 = d4_preds.get(name) or [[0]]
    a2 = greedy_preds.get(name) or [[0]]
    submission[name] = [{"attempt_1": a1, "attempt_2": a2}]
(RUN_DIR / "submission.json").write_text(json.dumps(submission))
print(f"Wrote {RUN_DIR/'submission.json'} | tasks={len(submission)}")


In [ ]:
# 8.4 — Bundle the run + (Colab) download
import shutil
zpath = ROOT / f"{RUN_ID}.zip"
shutil.make_archive(str(zpath).removesuffix(".zip"), "zip", RUN_DIR)
print("Bundle:", zpath, f"({zpath.stat().st_size/1e6:.1f} MB)")
try:
    from google.colab import files; files.download(str(zpath))
except Exception:
    print("Outside Colab — download from file browser:", zpath)


## What good looks like

- **Score history** should rise on each round and then plateau; the early-stop kicks in when 2 rounds in a row gain less than 1 pt.
- **Loss curves** should be: SFT decreasing fast (round 0), DPO loss decreasing slowly, GRPO reward rising.
- **Best round** is usually round 1 or 2 — heavy multi-round DPO+GRPO can over-collapse the policy. The "final" alias copies the best-round adapter so you don't have to chase that yourself.
- **Final D4-vote pass@2** in the **15–35% range on ARC-AGI-1 public eval** is realistic for this stack (Qwen3-14B + QLoRA + auto-loop). Anything above 25% is competitive with most published baselines.

If you want more, the next levers are: bigger base (Qwen3-32B with stronger quantization), TTT at inference time (notebook 06 Section 6), or longer training rounds (`HARD_PER_ROUND=500`, `N_ROUNDS=8`).
